Imports e configurações

In [1]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import date, timedelta
import random
import os
 
fake = Faker('pt_BR')
np.random.seed(42)
random.seed(42)

Dimensão Produtos

In [2]:
# dProdutos
categorias = {
    'Pães e Bisnaguinhas': {
        'linhas': ['Standard', 'Premium'],
        'marcas': ['Pullman', 'Wickbold'],
        'produtos': ['Pão de Forma 500g', 'Bisnaga 200g', 'Pão Integral 400g', 'Pão Hot Dog 300g'],
        'preco_base': (3.49, 7.99),
        'custo_pct': (0.45, 0.55)
    },
    'Bolos e Snacks': {
        'linhas': ['Standard', 'Premium'],
        'marcas': ['Bimbo', 'Pullman'],
        'produtos': ['Bolo Chocolate 250g', 'Bolinho Duplo 70g', 'Wafer Baunilha 130g', 'Pão de Mel 180g'],
        'preco_base': (2.99, 8.99),
        'custo_pct': (0.40, 0.50)
    },
    'Torradas e Wraps': {
        'linhas': ['Standard', 'Light'],
        'marcas': ['Nutrella', 'Wickbold'],
        'produtos': ['Torrada Traditional 160g', 'Wrap Integral 320g', 'Torrada Light 100g'],
        'preco_base': (4.99, 12.99),
        'custo_pct': (0.38, 0.48)
    }
}
 
produtos = []
pid = 1
for cat, info in categorias.items():
    for prod in info['produtos']:
        for marca in info['marcas']:
            preco = round(random.uniform(*info['preco_base']), 2)
            custo_pct = random.uniform(*info['custo_pct'])
            produtos.append({
                'id_produto': f'SKU-{pid:04d}',
                'nome_produto': prod,
                'categoria': cat,
                'linha': random.choice(info['linhas']),
                'marca': marca,
                'peso_g': random.choice([100, 130, 160, 180, 200, 250, 300, 320, 400, 500]),
                'preco_sugerido': preco,
                'custo_referencia': round(preco * custo_pct, 2),
                'ativo': True
            })
            pid += 1
 
df_produtos = pd.DataFrame(produtos)

Dimensão Canais

In [3]:
# dCanais
canais = [
    {'id_canal': 'CH-01', 'nome_canal': 'Varejo - Grande Porte', 'tipo_canal': 'Varejo', 'digital': False},
    {'id_canal': 'CH-02', 'nome_canal': 'Varejo - Pequeno Porte', 'tipo_canal': 'Varejo', 'digital': False},
    {'id_canal': 'CH-03', 'nome_canal': 'Food Service', 'tipo_canal': 'Food Service', 'digital': False},
    {'id_canal': 'CH-04', 'nome_canal': 'E-commerce', 'tipo_canal': 'E-commerce', 'digital': True},
    {'id_canal': 'CH-05', 'nome_canal': 'Atacado', 'tipo_canal': 'Atacado', 'digital': False},
]
df_canais = pd.DataFrame(canais)

Dimensão Regiões

In [4]:
# dRegioes
regioes = [
    {'id_regiao': 'REG-SP', 'nome_regiao': 'São Paulo', 'uf': 'SP', 'macrorregiao': 'Sudeste'},
    {'id_regiao': 'REG-RJ', 'nome_regiao': 'Rio de Janeiro', 'uf': 'RJ', 'macrorregiao': 'Sudeste'},
    {'id_regiao': 'REG-MG', 'nome_regiao': 'Minas Gerais', 'uf': 'MG', 'macrorregiao': 'Sudeste'},
    {'id_regiao': 'REG-RS', 'nome_regiao': 'Rio Grande do Sul', 'uf': 'RS', 'macrorregiao': 'Sul'},
    {'id_regiao': 'REG-PR', 'nome_regiao': 'Paraná', 'uf': 'PR', 'macrorregiao': 'Sul'},
    {'id_regiao': 'REG-BA', 'nome_regiao': 'Bahia', 'uf': 'BA', 'macrorregiao': 'Nordeste'},
    {'id_regiao': 'REG-PE', 'nome_regiao': 'Pernambuco', 'uf': 'PE', 'macrorregiao': 'Nordeste'},
    {'id_regiao': 'REG-GO', 'nome_regiao': 'Goiás', 'uf': 'GO', 'macrorregiao': 'Centro-Oeste'},
]
df_regioes = pd.DataFrame(regioes)

Dimensão Clientes

In [5]:
# dClientes
clientes = []
for i in range(1, 201):
    regiao = random.choice(df_regioes['id_regiao'].tolist())
    canal = random.choice(df_canais['id_canal'].tolist())
    segmentos = ['Varejo - Grande Porte', 'Varejo - Pequeno Porte', 'Restaurante', 'Padaria', 'Atacadista']
    clientes.append({
        'id_cliente': f'CLI-{i:04d}',
        'nome_cliente': fake.company(),
        'segmento': random.choice(segmentos),
        'cnpj': fake.cnpj(),
        'id_regiao': regiao,
        'id_canal': canal,
        'desde': fake.date_between(start_date=date(2018, 1, 1), end_date=date(2022, 12, 31))
    })
df_clientes = pd.DataFrame(clientes)

Tabela Fato (fVendas)

In [6]:
# ── Tabela Fato ───────────────────────────────────────────────────────────────
 
def gerar_sazonalidade(data):
    """Fatores sazonais por mês"""
    fatores = {1: 0.85, 2: 0.90, 3: 0.95, 4: 1.00, 5: 1.05,
               6: 1.10, 7: 1.08, 8: 1.05, 9: 1.00, 10: 1.10,
               11: 1.20, 12: 1.30}
    return fatores[data.month]
 
def gerar_tendencia(data):
    """Crescimento gradual ao longo dos 2 anos"""
    inicio = date(2023, 1, 1)
    dias = (data - inicio).days
    return 1 + (dias / 730) * 0.25  # +25% ao longo de 2 anos
 
registros = []
vid = 100001
 
datas = pd.date_range(start='2023-01-01', end='2024-12-31', freq='D')
 
for data in datas:
    # Quantidade de transações por dia varia
    n_transacoes = random.randint(30, 80)
    sazon = gerar_sazonalidade(data.date())
    tend = gerar_tendencia(data.date())
 
    for _ in range(n_transacoes):
        produto = df_produtos.sample(1).iloc[0]
        canal = random.choice(df_canais['id_canal'].tolist())
        regiao = random.choice(df_regioes['id_regiao'].tolist())
        cliente = df_clientes.sample(1).iloc[0]
 
        # Quantidade com sazonalidade e tendência
        qtd_base = random.uniform(100, 3000)
        qtd = round(qtd_base * sazon * tend + random.gauss(0, 50), 1)
        qtd = max(10, qtd)
 
        preco = produto['preco_sugerido']
        custo = produto['custo_referencia']
 
        # Desconto varia por canal
        descontos = {'CH-01': 0.08, 'CH-02': 0.03, 'CH-03': 0.12,
                     'CH-04': 0.05, 'CH-05': 0.15}
        desc_base = descontos.get(canal, 0.05)
        desconto = round(random.uniform(0, desc_base * 2), 3)
        desconto = min(desconto, 0.30)
 
        receita_bruta = round(qtd * preco, 2)
        custo_total = round(qtd * custo, 2)
        receita_liq = round(receita_bruta * (1 - desconto), 2)
        margem_bruta = round(receita_liq - custo_total, 2)
 
        registros.append({
            'id_venda': vid,
            'data_venda': data.date(),
            'id_produto': produto['id_produto'],
            'id_canal': canal,
            'id_regiao': regiao,
            'id_cliente': cliente['id_cliente'],
            'qtd_vendida': qtd,
            'preco_unit': preco,
            'custo_unit': custo,
            'receita_bruta': receita_bruta,
            'custo_total': custo_total,
            'desconto_pct': desconto,
            'receita_liq': receita_liq,
            'margem_bruta': margem_bruta
        })
        vid += 1
 
df_vendas = pd.DataFrame(registros)

Salva os arquivos e valida

In [8]:
# ── Salva os arquivos ─────────────────────────────────────────────────────────
os.makedirs('dados', exist_ok=True)
 
df_vendas.to_csv('dados/fVendas.csv', index=False)
df_produtos.to_csv('dados/dProdutos.csv', index=False)
df_canais.to_csv('dados/dCanais.csv', index=False)
df_regioes.to_csv('dados/dRegioes.csv', index=False)
df_clientes.to_csv('dados/dClientes.csv', index=False)
 
print(f"Base gerada com sucesso!")
print(f"   fVendas: {len(df_vendas):,} registros")
print(f"   dProdutos: {len(df_produtos)} produtos")
print(f"   dCanais: {len(df_canais)} canais")
print(f"   dRegioes: {len(df_regioes)} regiões")
print(f"   dClientes: {len(df_clientes)} clientes")

Base gerada com sucesso!
   fVendas: 40,423 registros
   dProdutos: 22 produtos
   dCanais: 5 canais
   dRegioes: 8 regiões
   dClientes: 200 clientes
